# Modelagem Preditiva de Reincidencia da Violencia Contra a Mulher

## Objetivo

Este notebook tem como objetivo desenvolver uma analise inicial para modelagem preditiva de reincidencia da violencia contra a mulher com base nos registros do SINAN/DATASUS.

A proposta e utilizar variaveis relacionadas ao perfil da vitima, caracteristicas da violencia, contexto da ocorrencia e informacoes sobre o agressor para identificar padroes associados a casos com maior probabilidade estatistica de reincidencia.

O problema sera tratado como uma tarefa de **classificacao binaria**.

## Variavel Alvo

A variavel alvo definida para o modelo e:

`OUT_VEZES`

Interpretacao adotada:

- `1` = houve reincidencia
- `0` = nao houve reincidencia

Assim, o modelo buscara aprender, a partir dos dados historicos, quais combinacoes de caracteristicas aparecem com maior frequencia em registros associados a reincidencia.

## Observacao Importante

O modelo nao deve ser interpretado como instrumento de diagnostico ou decisao automatica. Seu uso potencial e como apoio a triagem, priorizacao de acompanhamento e suporte a profissionais da rede de protecao, mantendo a decisao final sob responsabilidade humana.

## Imports

In [1]:
from pathlib import Path
import builtins

import numpy as np
import matplotlib.pyplot as plt
import shap
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

## Leitura da Base

Nesta etapa, a leitura segue uma versao simplificada da logica usada no script `01_selecionar_variaveis.py`:

- localizar o arquivo bruto disponivel no projeto;
- ler o dataset;
- filtrar apenas registros de mulheres (`CS_SEXO == "F"`);
- manter somente as colunas selecionadas para o projeto.

In [ ]:
COLUNAS = [
    "NU_IDADE_N",
    "CS_RACA",
    "CS_ESCOL_N",
    "SIT_CONJUG",
    "CS_GESTANT",
    "DEF_TRANS",
    "DEF_FISICA",
    "DEF_MENTAL",
    "DEF_VISUAL",
    "DEF_AUDITI",
    "TRAN_MENT",
    "SG_UF",
    "LOCAL_OCOR",
    "VIOL_FISIC",
    "VIOL_PSICO",
    "VIOL_SEXU",
    "VIOL_NEGLI",
    "VIOL_FINAN",
    "VIOL_TORT",
    "LES_AUTOP",
    "AG_FORCA",
    "AG_CORTE",
    "AG_FOGO",
    "AG_AMEACA",
    "REL_CONJ",
    "REL_EXCON",
    "REL_NAMO",
    "REL_PAI",
    "REL_MAE",
    "AUTOR_SEXO",
    "AUTOR_ALCO",
    "NUM_ENVOLV",
    "OUT_VEZES",
]

raiz_projeto = Path("..").resolve()
arquivo_csv = raiz_projeto / "data" / "raw" / "VIOLBR25_ptbr.csv"
arquivo_xlsx = raiz_projeto / "data" / "raw" / "VIOLBR25.xlsx"

if arquivo_csv.exists():
    df_bruto = pd.read_csv(arquivo_csv, sep=";", low_memory=False)
elif arquivo_xlsx.exists():
    df_bruto = pd.read_excel(arquivo_xlsx)
else:
    raise FileNotFoundError("Nenhum arquivo bruto encontrado em data/raw.")

df = df_bruto[df_bruto["CS_SEXO"] == "F"].copy()
df = df[COLUNAS].copy()

print(f"Registros: {len(df):,}")
print(f"Colunas: {len(df.columns)}")
df.head()

## Diagnostico de Nulos e Inconsistencias

Antes de limpar a base, vale verificar nulos, codigos suspeitos e a distribuicao inicial da variavel alvo.

In [ ]:
codigos_suspeitos = {9, 99, 999, 9999, "9", "99", "999", "9999", "Ignorado", "Nao informado", "Em branco", ""}

resumo_qualidade = pd.DataFrame({
    "coluna": df.columns,
    "tipo": [str(df[coluna].dtype) for coluna in df.columns],
    "qtd_nulos": [df[coluna].isna().sum() for coluna in df.columns],
    "pct_nulos": [round(df[coluna].isna().mean() * 100, 2) for coluna in df.columns],
    "qtd_valores_unicos": [df[coluna].nunique(dropna=True) for coluna in df.columns],
    "qtd_codigos_suspeitos": [df[coluna].isin(codigos_suspeitos).sum() for coluna in df.columns],
})

resumo_qualidade.sort_values(by=["pct_nulos", "qtd_codigos_suspeitos"], ascending=False).head(15)

In [ ]:
df["OUT_VEZES"].value_counts(dropna=False)

## Tratamento Inicial da Base

Agora convertemos codigos de ignorado para `NaN` e tratamos `OUT_VEZES` como alvo binario.

In [ ]:
CODIGOS_IGNORADOS = {
    "CS_RACA": [9],
    "CS_ESCOL_N": [9],
    "SIT_CONJUG": [9],
    "CS_GESTANT": [9],
    "DEF_TRANS": [9],
    "DEF_FISICA": [9],
    "DEF_MENTAL": [9],
    "DEF_VISUAL": [9],
    "DEF_AUDITI": [9],
    "TRAN_MENT": [9],
    "LOCAL_OCOR": [9, 99],
    "VIOL_FISIC": [9],
    "VIOL_PSICO": [9],
    "VIOL_SEXU": [9],
    "VIOL_NEGLI": [9],
    "VIOL_FINAN": [9],
    "VIOL_TORT": [9],
    "LES_AUTOP": [9],
    "AG_FORCA": [9],
    "AG_CORTE": [9],
    "AG_FOGO": [9],
    "AG_AMEACA": [9],
    "REL_CONJ": [9],
    "REL_EXCON": [9],
    "REL_NAMO": [9],
    "REL_PAI": [9],
    "REL_MAE": [9],
    "AUTOR_SEXO": [9],
    "AUTOR_ALCO": [9],
    "NUM_ENVOLV": [9],
}

MAPA_ALVO = {1: 1, 2: 0}

In [ ]:
df_tratado = df.copy()

for coluna, codigos in CODIGOS_IGNORADOS.items():
    if coluna in df_tratado.columns:
        df_tratado[coluna] = df_tratado[coluna].replace(codigos, pd.NA)

df_tratado = df_tratado[df_tratado["OUT_VEZES"].isin(MAPA_ALVO.keys())].copy()
df_tratado["OUT_VEZES"] = df_tratado["OUT_VEZES"].map(MAPA_ALVO).astype("int64")

print(f"Registros antes do tratamento: {len(df):,}")
print(f"Registros depois do tratamento: {len(df_tratado):,}")
df_tratado.head()

### Verificacao da Variavel Alvo Apos o Tratamento

In [ ]:
df_tratado["OUT_VEZES"].value_counts(dropna=False)

### Resumo de Nulos Apos o Tratamento Inicial

In [ ]:
resumo_pos_tratamento = pd.DataFrame({
    "coluna": df_tratado.columns,
    "qtd_nulos": [df_tratado[coluna].isna().sum() for coluna in df_tratado.columns],
    "pct_nulos": [round(df_tratado[coluna].isna().mean() * 100, 2) for coluna in df_tratado.columns],
})

resumo_pos_tratamento.sort_values(by="pct_nulos", ascending=False).head(15)

## Limpeza Fina

Nesta etapa, fazemos uma revisao adicional antes da modelagem.

In [ ]:
IDADE_MINIMA = 0
IDADE_MAXIMA = 120
LIMITE_NULOS_EXCLUSAO = 30.0

df_limpo = df_tratado.copy()

if "NU_IDADE_N" in df_limpo.columns:
    mascara_idade_invalida = (
        (df_limpo["NU_IDADE_N"] < IDADE_MINIMA)
        | (df_limpo["NU_IDADE_N"] > IDADE_MAXIMA)
    )
    df_limpo.loc[mascara_idade_invalida, "NU_IDADE_N"] = pd.NA

percentuais_nulos = (df_limpo.isna().mean() * 100).round(2)
colunas_removidas = percentuais_nulos[percentuais_nulos > LIMITE_NULOS_EXCLUSAO].index.tolist()
df_limpo = df_limpo.drop(columns=colunas_removidas, errors="ignore")

print(f"Colunas removidas: {colunas_removidas if colunas_removidas else 'nenhuma'}")
print(f"Formato final da base limpa: {df_limpo.shape}")

In [ ]:
pd.DataFrame({
    "coluna": percentuais_nulos.index,
    "pct_nulos": percentuais_nulos.values,
    "removida": percentuais_nulos.index.isin(colunas_removidas),
}).sort_values(by="pct_nulos", ascending=False).head(15)

## Definicao do Problema e Divisao Treino/Teste

In [ ]:
X = df_limpo.drop(columns=["OUT_VEZES"]).copy()
y = df_limpo["OUT_VEZES"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

In [ ]:
resumo_divisao = pd.DataFrame([
    {
        "registros_treino": len(X_train),
        "registros_teste": len(X_test),
        "classe_0_treino": int((y_train == 0).sum()),
        "classe_1_treino": int((y_train == 1).sum()),
        "classe_0_teste": int((y_test == 0).sum()),
        "classe_1_teste": int((y_test == 1).sum()),
    }
])

resumo_divisao

## Preparacao Final para o Modelo

In [ ]:
print("Nulos em X_train:", X_train.isna().sum().sum())
print("Nulos em X_test:", X_test.isna().sum().sum())
print("Classes y_train:\n", y_train.value_counts(dropna=False).sort_index())
print("Classes y_test:\n", y_test.value_counts(dropna=False).sort_index())

In [ ]:
X_train_modelo = X_train.copy()
X_test_modelo = X_test.copy()

for coluna in X_train_modelo.columns:
    if coluna == "NU_IDADE_N":
        valor_preenchimento = X_train_modelo[coluna].median()
    else:
        moda = X_train_modelo[coluna].mode(dropna=True)
        valor_preenchimento = moda.iloc[0] if not moda.empty else 0

    X_train_modelo[coluna] = X_train_modelo[coluna].fillna(valor_preenchimento)
    X_test_modelo[coluna] = X_test_modelo[coluna].fillna(valor_preenchimento)

print("Nulos em X_train_modelo:", X_train_modelo.isna().sum().sum())
print("Nulos em X_test_modelo:", X_test_modelo.isna().sum().sum())

## Modelagem com Arvore de Decisao

In [ ]:
modelo_arvore = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,
)

modelo_arvore.fit(X_train_modelo, y_train)

In [ ]:
y_pred = modelo_arvore.predict(X_test_modelo)

pd.DataFrame({
    "y_real": y_test.reset_index(drop=True),
    "y_pred": pd.Series(y_pred).reset_index(drop=True),
}).head(10)

In [ ]:
metricas_arvore = pd.DataFrame([
    {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred), 4),
    }
])

metricas_arvore

In [ ]:
relatorio_classificacao = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
).transpose()

relatorio_classificacao

In [ ]:
matriz = confusion_matrix(y_test, y_pred)

figura, eixo = plt.subplots(figsize=(6, 5))
display = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Nao reincidencia", "Reincidencia"],
)
display.plot(ax=eixo, colorbar=False)
eixo.set_title("Matriz de Confusao - Arvore de Decisao")
plt.show()

In [ ]:
importancias = pd.DataFrame({
    "variavel": X_train_modelo.columns,
    "importancia": modelo_arvore.feature_importances_,
}).sort_values(by="importancia", ascending=False)

importancias.head(10)

## Explicabilidade com SHAP - Arvore de Decisao

O SHAP ajuda a mostrar quais variaveis empurram a previsao para reincidencia ou nao reincidencia em cada observacao.


In [ ]:
amostra_teste_arvore = X_test_modelo.sample(n=builtins.min(1000, len(X_test_modelo)), random_state=42).copy()
amostra_teste_arvore = amostra_teste_arvore.apply(pd.to_numeric, errors="coerce").astype("float64")

explainer_arvore = shap.TreeExplainer(modelo_arvore)
shap_values_arvore = explainer_arvore.shap_values(amostra_teste_arvore)

if isinstance(shap_values_arvore, list):
    shap_classe_1_arvore = np.asarray(shap_values_arvore[1], dtype="float64")
elif getattr(shap_values_arvore, "ndim", 0) == 3:
    shap_classe_1_arvore = np.asarray(shap_values_arvore[:, :, 1], dtype="float64")
else:
    shap_classe_1_arvore = np.asarray(shap_values_arvore, dtype="float64")

shap.summary_plot(shap_classe_1_arvore, amostra_teste_arvore)


In [ ]:
shap_resumo_arvore = pd.DataFrame({
    "variavel": amostra_teste_arvore.columns,
    "mean_abs_shap": abs(shap_classe_1_arvore).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

shap_resumo_arvore.head(10)


In [ ]:
top_importancias = importancias.head(10).iloc[::-1]

plt.figure(figsize=(10, 6))
plt.barh(top_importancias["variavel"], top_importancias["importancia"], color="steelblue")
plt.title("Top 10 Variaveis Mais Importantes")
plt.xlabel("Importancia")
plt.ylabel("Variavel")
plt.show()

In [ ]:
plt.figure(figsize=(22, 12))
plot_tree(
    modelo_arvore,
    feature_names=X_train_modelo.columns,
    class_names=["Nao reincidencia", "Reincidencia"],
    filled=True,
    rounded=True,
    fontsize=7,
)
plt.title("Arvore de Decisao")
plt.show()

## Modelagem com Regressao Logistica

Agora aplicamos um segundo modelo de classificacao para comparar com a Arvore de Decisao.

In [ ]:
modelo_logistico = LogisticRegression(
    random_state=42,
    max_iter=1000,
)

modelo_logistico.fit(X_train_modelo, y_train)

## Previsoes da Regressao Logistica

Depois do treino, o modelo gera previsoes para o conjunto de teste. Aqui conseguimos comparar os valores reais com os valores previstos.

In [ ]:
y_pred_log = modelo_logistico.predict(X_test_modelo)

pd.DataFrame({
    "y_real": y_test.reset_index(drop=True),
    "y_pred_log": pd.Series(y_pred_log).reset_index(drop=True),
}).head(10)

## Metricas da Regressao Logistica

As mesmas metricas da Arvore de Decisao sao usadas aqui para facilitar a comparacao entre os modelos.

In [ ]:
metricas_logistica = pd.DataFrame([
    {
        "accuracy": round(accuracy_score(y_test, y_pred_log), 4),
        "recall": round(recall_score(y_test, y_pred_log), 4),
        "f1_score": round(f1_score(y_test, y_pred_log), 4),
    }
])

metricas_logistica

## Relatorio de Classificacao

Esse relatorio mostra como a Regressao Logistica se comportou em cada classe, separando precision, recall e f1-score.

In [ ]:
relatorio_logistica = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_log,
        output_dict=True,
        zero_division=0,
    )
).transpose()

relatorio_logistica

## Matriz de Confusao - Regressao Logistica

A matriz de confusao ajuda a ver quantos acertos e erros aconteceram em cada classe.

In [ ]:
matriz_log = confusion_matrix(y_test, y_pred_log)

figura, eixo = plt.subplots(figsize=(6, 5))
display = ConfusionMatrixDisplay(
    confusion_matrix=matriz_log,
    display_labels=["Nao reincidencia", "Reincidencia"],
)
display.plot(ax=eixo, colorbar=False)
eixo.set_title("Matriz de Confusao - Regressao Logistica")
plt.show()

## Coeficientes Mais Relevantes

Na Regressao Logistica, os coeficientes mostram quais variaveis tiveram mais peso na previsao. Aqui usamos o valor absoluto para destacar a forca da influencia.

In [ ]:
coeficientes_log = pd.DataFrame({
    "variavel": X_train_modelo.columns,
    "coeficiente": modelo_logistico.coef_[0],
    "coeficiente_absoluto": abs(modelo_logistico.coef_[0]),
}).sort_values(by="coeficiente_absoluto", ascending=False)

coeficientes_log.head(10)

## Explicabilidade com SHAP - Regressao Logistica

Na regressao logistica, o SHAP ajuda a complementar a leitura dos coeficientes mostrando o impacto das variaveis nas previsoes individuais.


In [ ]:
amostra_treino_log = X_train_modelo.sample(n=builtins.min(100, len(X_train_modelo)), random_state=42).copy()
amostra_teste_log = X_test_modelo.sample(n=builtins.min(1000, len(X_test_modelo)), random_state=42).copy()

amostra_treino_log = amostra_treino_log.apply(pd.to_numeric, errors="coerce").astype("float64")
amostra_teste_log = amostra_teste_log.apply(pd.to_numeric, errors="coerce").astype("float64")

explainer_log = shap.LinearExplainer(modelo_logistico, amostra_treino_log)
shap_values_log = explainer_log.shap_values(amostra_teste_log)

if isinstance(shap_values_log, list):
    shap_classe_1_log = np.asarray(shap_values_log[1], dtype="float64")
elif getattr(shap_values_log, "ndim", 0) == 3:
    shap_classe_1_log = np.asarray(shap_values_log[:, :, 1], dtype="float64")
else:
    shap_classe_1_log = np.asarray(shap_values_log, dtype="float64")

shap.summary_plot(shap_classe_1_log, amostra_teste_log)


In [ ]:
shap_resumo_log = pd.DataFrame({
    "variavel": amostra_teste_log.columns,
    "mean_abs_shap": abs(shap_classe_1_log).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

shap_resumo_log.head(10)


In [ ]:
top_coeficientes = coeficientes_log.head(10).iloc[::-1]

plt.figure(figsize=(10, 6))
plt.barh(top_coeficientes["variavel"], top_coeficientes["coeficiente_absoluto"], color="darkorange")
plt.title("Top 10 Variaveis Mais Relevantes - Regressao Logistica")
plt.xlabel("Coeficiente absoluto")
plt.ylabel("Variavel")
plt.show()

## Comparacao Entre os Modelos

Nesta ultima etapa, colocamos Arvore de Decisao e Regressao Logistica lado a lado para ver qual teve melhor desempenho.

In [ ]:
comparacao_modelos = pd.DataFrame([
    {
        "modelo": "Arvore de Decisao",
        "accuracy": metricas_arvore.loc[0, "accuracy"],
        "recall": metricas_arvore.loc[0, "recall"],
        "f1_score": metricas_arvore.loc[0, "f1_score"],
    },
    {
        "modelo": "Regressao Logistica",
        "accuracy": metricas_logistica.loc[0, "accuracy"],
        "recall": metricas_logistica.loc[0, "recall"],
        "f1_score": metricas_logistica.loc[0, "f1_score"],
    }
])

comparacao_modelos